## ▶ Colab setup — GitHub repo + dataset from Google Drive

Run this cell **first** on Google Colab. It clones the GitHub repo, installs the
`phytolabs` package, mounts your Drive, and unzips + reshapes the Leaf-rust
dataset into `data/{train,val}/{healthy,rust}` so the rest of the notebook runs
on **real data**. Edit `ZIP_PATH` if your zip lives elsewhere in Drive.

This notebook is **standalone**: if the Stage-1 feature tables from notebook 02
aren't present (a fresh Colab runtime), it regenerates them from the data.
On a non-Colab machine this cell is a harmless no-op.

In [ ]:
# === Colab setup: GitHub repo + dataset from Google Drive ===================
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Adian17/PhytoLabs.git"
ZIP_PATH = "/content/drive/MyDrive/ece186/WheatLeafRust.zip"  # <-- adjust if needed

if "google.colab" in sys.modules:
    # 1) Clone the repo + install the package.
    if not os.path.isdir("/content/PhytoLabs"):
        !git clone -q $REPO_URL /content/PhytoLabs
    %cd /content/PhytoLabs
    !pip -q install -e .
    # Editable install / _setup aren't importable mid-kernel; add paths explicitly.
    for _p in ("/content/PhytoLabs/src", "/content/PhytoLabs/notebooks"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 2) Mount Drive + unzip the dataset (only the first time).
    if not (Path("data/raw").exists() and any(Path("data/raw").iterdir())):
        from google.colab import drive
        drive.mount("/content/drive")
        !rm -rf data/raw && mkdir -p data/raw
        !unzip -q "$ZIP_PATH" -d data/raw

    # 3) Reshape control/diseased -> data/{train,val}/{healthy,rust} (only once).
    if not (Path("data/train/rust").exists() and any(Path("data/train/rust").glob("*"))):
        raw = Path("data/raw")
        def _find(name):
            cands = [d for d in raw.rglob("*") if d.is_dir() and d.name.lower() == name]
            if not cands:
                raise FileNotFoundError(f"No '{name}' folder under data/raw — check the zip layout.")
            train_cands = [d for d in cands if "train" in str(d).lower()]
            return str((train_cands or cands)[0])
        H, R = _find("control"), _find("diseased")
        print("healthy <-", H, "\nrust    <-", R)
        !python -m scripts.reshape_data --healthy-src "$H" --rust-src "$R" --out data --val-fraction 0.2
    print("dataset:", {c: len(list(Path("data/train", c).glob("*"))) for c in ("healthy", "rust")})

# Stage 2 — Logistic Regression + SGD (from scratch)

Train a logistic regression in pure NumPy (sigmoid + BCE + L2, mini-batch SGD) on the Stage 1 features, then evaluate image-level performance and calibration. A **0.45-0.55** band defines a third **suspicious** class.

In [ ]:
from _setup import ARTIFACTS_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs.logreg import LogisticRegressionSGD
from phytolabs.features import FEATURE_NAMES
from phytolabs import segmentation, pipeline, viz, metrics, calibration

def _regenerate_features():
    """Rebuild features_{train,val}.npz from data so this notebook is standalone.

    On Colab each notebook gets a fresh runtime, so the artifacts from
    notebooks 01/02 won't be present — regenerate them here if missing.
    """
    data_dir = ensure_dataset()
    gmm_path = ARTIFACTS_DIR / 'gmm.joblib'
    if gmm_path.exists():
        leaf_gmm = segmentation.LeafGMM.load(gmm_path)
    else:
        leaf_gmm = pipeline.fit_gmm_from_dir(data_dir / 'train', k=4, limit_per_class=150)
        leaf_gmm.save(gmm_path)
    for split in ('train', 'val'):
        Xs, ys, ps = pipeline.build_feature_table(data_dir / split, leaf_gmm)
        np.savez(ARTIFACTS_DIR / f'features_{split}.npz', X=Xs, y=ys, paths=np.array(ps, dtype=object))

def load_table(split):
    path = ARTIFACTS_DIR / f'features_{split}.npz'
    if not path.exists():
        print('feature tables not found — regenerating from data...')
        _regenerate_features()
    d = np.load(path, allow_pickle=True)
    return d['X'], d['y']

X_train, y_train = load_table('train')
X_val, y_val = load_table('val')
print('train', X_train.shape, '| val', X_val.shape)

## Train

In [ ]:
model = LogisticRegressionSGD(lr=0.1, epochs=300, batch_size=16, l2=1e-3, random_state=0)
model.fit(X_train, y_train)
for name, w in zip(FEATURE_NAMES, model.w):
    print(f'{name:22s} {w:+.3f}')
print(f'{"bias":22s} {model.b:+.3f}')

In [ ]:
viz.plot_loss(model.loss_history)
plt.show()

## Evaluate on the validation split

In [ ]:
proba = model.predict_proba(X_val)
pred = model.predict(X_val)
report = metrics.classification_report(y_val, pred, proba)
report

In [ ]:
viz.plot_confusion_matrix(y_val, pred); plt.show()
viz.plot_roc(y_val, proba); plt.show()

## Calibration and the suspicious band

Reliability diagram + how many val images land in each band class.

In [ ]:
viz.plot_reliability(y_val, proba, n_bins=5); plt.show()
print('ECE:', calibration.expected_calibration_error(y_val, proba))
print('band summary:', calibration.band_summary(proba, 0.45, 0.55))

## Persist the trained classifier

In [ ]:
model.save(ARTIFACTS_DIR / 'logreg.joblib')
print('saved', ARTIFACTS_DIR / 'logreg.joblib')